# Encoding-Invariant Search for Connected Line-Graph Fragments

A set of Hamiltonian terms whose frustration graph — one vertex per term, one edge
per anticommuting pair — is a connected line graph forms a free-fermion-solvable
grouping. This notebook implements the search for such fragments laid out in the
pseudocode procedures `Supports`, `Expand`, `Order`, and `Search`, and reproduces
the two tables that report it on four molecular Hamiltonians (H3+, H4, LiH, and
H2O; STO-3G spin singlets from the Symmer test data): the structure table
(`tab:sys-db-1body`) and the search table (`tab:line-graph-search`).

Every decision of the search is taken on the *Majorana supports* of the terms,
which do not depend on the fermion-to-qubit encoding of the Hamiltonian; the term
coefficients enter only through one of the two visiting orders and through the
final ranking. The sections below follow the four procedures, build the two tables
from the Jordan-Wigner files, and close by repeating both on the Bravyi-Kitaev
files.

In [ ]:
const offlineMode::Bool = false

#> Reproducible environment: a dedicated project (`Project.toml` and `Manifest.toml`)
#> is created next to this notebook, and the exact package versions pinned below are
#> re-applied on every run, so the environment always matches this notebook version;
#> roll back the notebook itself to reproduce older results with their own pins.
import Pkg

#> The 1.12 floor also covers the lexicographic `isless` on vectors that the support
#> keys, the ranking tuples, and the support fingerprints below rely on
@assert VERSION >= v"1.12" "This notebook targets Julia 1.12+."

Pkg.activate(@__DIR__) #> Under IJulia, `@__DIR__` is the notebook's working directory

if offlineMode && isfile(joinpath(@__DIR__, "Manifest.toml"))
    Pkg.instantiate() #> Materialize the existing `Manifest.toml` without any re-check
else #> Also the fallback when `offlineMode = true` finds no `Manifest.toml` yet
    #> Undo pins left by a previous (possibly older) notebook run so the `Pkg.add`
    #> calls below can also move the pinned versions when a newer notebook bumps them
    if isfile(joinpath(@__DIR__, "Manifest.toml"))
        for dep in values(Pkg.dependencies())
            dep.is_pinned && Pkg.free(dep.name)
        end
    end

    Pkg.add(name="JSON", version="0.21.4")
    Pkg.add(name="Paulimorphic", version=v"0.2.0")
    Pkg.pin(["JSON"])
end

using Paulimorphic
using Paulimorphic: neumaierSum, evalMagnitude
using Printf: @sprintf
import JSON
import Downloads

## The Data Layer

The Hamiltonian files come from the Symmer repository's test data and are cached
next to this notebook. `loadSymmerHam` parses a file into a `PauliSum`, validating
that every coefficient is real; called with `excludeIdentity=true`, it removes the
identity term, so that the term count `|V|` and every l1 norm below are
identity-excluded. `genSymmerMajFrame` returns the `2n` Majorana operators of the
file's encoding for `n` modes in the blocked layout
`[γ_1, γ_3, ..., γ_{2n-1}, γ_2, γ_4, ..., γ_{2n}]`: the two operators of mode `p`,
`γ_{2p-1}` and `γ_{2p}`, sit at positions `p` and `n + p`.

In [2]:
const CONSTVAR!!symmerRawUrlBase = "https://raw.githubusercontent.com/qmatter-labs/" *
                                   "symmer/main/tests/hamiltonian_data/"

const CONSTVAR!!systemNames = ["H3+", "H4", "LiH", "H2O"]

"""
    fetchSymmerHamFiles(dir::AbstractString, names::AbstractVector{<:AbstractString},
                        overwrite::Bool=false) -> Vector{String}

Ensure the Symmer Hamiltonian files `names` exist under `dir` (downloading any that are
missing, or all of them under `overwrite`) and return their paths in the given order.
"""
function fetchSymmerHamFiles(dir::AbstractString, names::AbstractVector{<:AbstractString},
                             overwrite::Bool=false)
    mkpath(dir)
    map(names) do name
        path = joinpath(dir, name)
        if overwrite || !isfile(path)
            Downloads.download(CONSTVAR!!symmerRawUrlBase * name, path)
        end
        path
    end
end

"""
    loadSymmerHam(path::AbstractString; excludeIdentity::Bool=false) -> PauliSum{Float64}

Load a Symmer Hamiltonian JSON file (a top-level `"hamiltonian"` object mapping Pauli
labels to `[real, imaginary]` coefficient pairs) into a `PauliSum`, validating that
every imaginary part is exactly zero. `excludeIdentity` (opt-in) removes the identity
term after construction, so every downstream count and coefficient-ratio denominator
is identity-excluded.
"""
function loadSymmerHam(path::AbstractString; excludeIdentity::Bool=false)
    hamEntry = get(JSON.parsefile(path), "hamiltonian", missing)
    if !(hamEntry isa AbstractDict)
        throw(ArgumentError("The file at `path` must contain a top-level " *
                            "\"hamiltonian\" JSON object."))
    end

    strs = PauliStr[]
    coeffs = Float64[]

    for (label, value) in hamEntry
        if !(value isa AbstractVector && length(value) == 2 && all(Base.Fix2(isa, Real),
                                                                   value))
            throw(ArgumentError("The coefficient of term \"$label\" must be a " *
                                "two-element (real, imaginary) array."))
        end
        imagPart = value[begin+1]
        iszero(imagPart) ||
        throw(ArgumentError("The coefficient of term \"$label\" must be real; got " *
                            "imaginary part $imagPart."))
        push!(strs, toPauliStr(label))
        push!(coeffs, Float64(value[begin]))
    end

    ham = PauliSum(strs, coeffs)
    #> The filtered sum keeps the canonical order of `ham` and shares its strings
    excludeIdentity ? PauliSum(term -> !isIdentity(term.first), ham) : ham
end

"""
    genSymmerMajFrame(name::AbstractString, nMode::Integer) -> Vector{PauliStr}

Return the Majorana operators of the encoding named by the suffix of `name` (`"_JW"` or
`"_BK"`) for `nMode` modes, in the blocked layout
`[γ_1, γ_3, ..., γ_{2n-1}, γ_2, γ_4, ..., γ_{2n}]` with `n = nMode`: the two operators
of mode `p`, `γ_{2p-1}` and `γ_{2p}`, occupy positions `p` and `p + nMode`. Throws for
any other suffix.
"""
function genSymmerMajFrame(name::AbstractString, nMode::Integer)
    enc = if endswith(name, "_JW")
        genJordanWignerEnc(nMode)
    elseif endswith(name, "_BK")
        genBravyiKitaevEnc(nMode)
    else
        throw(ArgumentError("`name` must end with \"_JW\" or \"_BK\" to select an " *
                            "encoding; got \"$name\"."))
    end
    vcat(enc.first, enc.second)
end

genSymmerMajFrame

## Majorana Supports (`Supports`)

Up to a phase, every Pauli string on `n` qubits is the product of a unique subset
of the `2n` Majorana operators; the index set of that subset is the term's
*support*. Anticommutation alone recovers it: `γ_q` anticommutes with the product
over a support `S` iff `|S| + (q ∈ S)` is odd, so the operators that anticommute
with a term mark `S` itself when their number is even and its complement when
odd. `genMajoranaMasks` returns each support as a `BitVector` (its
*mask*) in the frame layout and, under `sanityCheck`, re-multiplies it to confirm
the term. Two terms with supports `S` and `T` anticommute iff `|S||T| − |S ∩ T|` is
odd, so the frustration graph is a function of the supports.

Terms are ordered by the support key `κ = (|S|, mask)`, the mask read as a binary
number with position 1 as its lowest bit and compared in the code as
`(sum(mask), reverse(mask))`. In the frame layout this is the pseudocode's
`κ(r) = (|S_r|, Σ_{2p-1 ∈ S_r} 2^{p-1} + Σ_{2p ∈ S_r} 2^{n+p-1})`. Two sets of
supports are singled out. The *diagonal block* `S_diag` holds the paired supports,
which contain `γ_{2p-1}` and `γ_{2p}` together or not at all; for the encodings
used here these are exactly the diagonal (`I`/`Z`-only) terms, an identification
the search asserts. The *quadratic sector* `S_quad` holds the supports of size two.
The two sets overlap in the occupation terms, whose supports are `{2p-1, 2p}`.

In [3]:
#>-- Reference(s) --<#
#> [DOI] 10.22331/q-2020-06-04-278 #>> Chapman & Flammia: Majorana monomial bases
"""
    genMajoranaMasks(ham::PauliSum, majFrame::AbstractVector{PauliStr},
                     sanityCheck::Bool=true) -> Vector{BitVector}

Return the Majorana support of every term of `ham` with respect to `majFrame` in the
blocked layout of [`genSymmerMajFrame`](@ref): entry `j` of the `i`-th returned mask
marks whether `majFrame[j]` appears in the frame-product decomposition of the `i`-th
term. Each support is recovered from the anticommutation pattern alone: `majFrame[j]`
anticommutes with a product over `S` iff `|S| + (j ∈ S)` is odd, so the marked pattern
equals the indicator of `S` when its weight is even and the complement when odd.
`sanityCheck` (opt-out) validates the frame via `checkMajoranaEnc`, re-multiplies every
decomposition, and compares it with the stored string up to a phase. Distinct terms
must carry distinct supports, which makes every mask-keyed ordering below total.
"""
function genMajoranaMasks(ham::PauliSum, majFrame::AbstractVector{PauliStr},
                          sanityCheck::Bool=true)
    iseven(length(majFrame)) ||
    throw(ArgumentError("`majFrame` must hold an even number of operators."))
    nMode = length(majFrame) ÷ 2
    if sanityCheck #> Throws unless the two halves of the frame form a Majorana encoding
        checkMajoranaEnc((@view majFrame[begin:(begin+nMode-1)]) =>
                         (@view majFrame[(begin+nMode):end]), true)
    end
    all(str -> countSites(str) == nMode, ham.str) ||
    throw(ArgumentError("`majFrame` must span the same site count as `ham`."))
    masks = map(_ -> falses(2nMode), 1:countTerms(ham))

    for (i, (str, mask)) in enumerate(zip(ham.str, masks))
        for (j, op) in enumerate(majFrame)
            mask[begin+j-1] = checkAntiCom(str, op)
        end

        #> An odd number of marks records the operators absent from the support
        isodd(sum(mask)) && (mask .⊻= true)

        if sanityCheck
            majStr = PauliStr(nMode)
            for (pos, occ) in enumerate(mask)
                occ && (majStr *= majFrame[begin+pos-1])
            end

            if !compareSites(majStr, str)
                throw(AssertionError("The support reconstruction failed for term $i " *
                                     "of `ham`."))
            end
        end
    end

    allunique(masks) || throw(ArgumentError("`ham` should not have mergeable terms."))

    masks
end

"""
    listDiagBlock(masks::AbstractVector{BitVector}) -> Vector{Int}

Indices of the *paired* supports in ascending order: the two halves of the blocked mask
agree, i.e., `γ_{2p-1}` and `γ_{2p}` enter together or not at all for every mode `p`.
For Jordan-Wigner- and Bravyi-Kitaev-style encodings, the terms with a non-empty paired
support are exactly the diagonal (`I`/`Z`-only) strings; the empty support of an
identity term is paired as well, which is why the search requires its absence.
"""
function listDiagBlock(masks::AbstractVector{BitVector})
    isPaired(mask::BitVector) = (@view mask[begin:(end÷2)]) == (@view mask[(end÷2+1):end])
    findall(isPaired, masks)
end

"""
    listQuadSector(masks::AbstractVector{BitVector}) -> Vector{Int}

Indices of the supports of size two, in ascending order: the terms that are, up to a
phase, products of two distinct Majorana operators.
"""
listQuadSector(masks::AbstractVector{BitVector}) = findall(mask -> sum(mask) == 2, masks)

listQuadSector

## Greedy Expansion (`Expand`)

`expandConnectedLineGraph` extends a fragment in one pass over a candidate
sequence. A candidate with no frustration edge into the (non-empty) fragment is
*deferred*: admitting it now would disconnect the graph, but it stays eligible for
later passes. A candidate whose admission keeps the frustration graph a connected
line graph is accepted. Any other candidate is discarded for good: line graphs are
closed under taking induced subgraphs, so a term that breaks the property once
breaks it in every later extension of the fragment. `isConnectedLineGraph` decides
membership through the library's root-graph reconstruction, and
`listConnectedComponents` splits a term set into the connected components of its
frustration graph.

In [4]:
"""
    checkTermIds(strs::AbstractVector{PauliStr}, ids::AbstractVector{Int}) -> Nothing

Validate that every entry of `ids` indexes into `strs` and that `ids` holds no
duplicates. Applied to the concatenation of a fragment and a candidate sequence, the
duplicate check doubles as a disjointness guarantee.
"""
function checkTermIds(strs::AbstractVector{PauliStr}, ids::AbstractVector{Int})
    scope = eachindex(strs)
    for idx in ids
        if !(idx in scope)
            throw(DomainError(idx, "Every entry of `ids` must be a valid index into " *
                                   "`strs`."))
        end
    end
    allunique(ids) || throw(ArgumentError("`ids` must not contain duplicate entries."))

    nothing
end

#>-- Reference(s) --<#
#> [DOI] 10.1145/321850.321853 #>> Lehot: root-graph recovery behind `genRootGraph`
"""
    isConnectedLineGraph(g::StrictGraph, explicitError::Bool=false) -> Bool

Whether `g` is a connected line graph, decided by the library's root-graph
reconstruction. `explicitError` (opt-in) escalates a negative verdict to an
`ArgumentError`, turning the predicate into an assertion. The empty graph passes.
"""
function isConnectedLineGraph(g::StrictGraph, explicitError::Bool=false)
    #> `genRootGraph` need not re-check the connectivity established by the first test
    res = length(listComponents(g)) <= 1 &&
          first(genRootGraph(g, checkConnectivity=false))
    explicitError && !res && throw(ArgumentError("`g` is not a connected line graph."))

    res
end

"""
    listConnectedComponents(strs::AbstractVector{PauliStr},
                            termIds::AbstractVector{Int}) -> Vector{Vector{Int}}

Split the frustration graph induced on the terms `termIds` (vertices = the given terms,
edges = anticommuting pairs among them) into its connected components. Each component
lists its terms in the order of `termIds`, and the components are ordered by their
first terms in `termIds`.
"""
function listConnectedComponents(strs::AbstractVector{PauliStr},
                                 termIds::AbstractVector{Int})
    checkTermIds(strs, termIds)
    edges = getFrustrationInfo(strs[termIds]).second
    components = listComponents(StrictGraph(length(termIds), edges))
    [termIds[labels] for labels in components]
end

"""
    expandConnectedLineGraph(strs::AbstractVector{PauliStr},
                             fragIds::AbstractVector{Int},
                             candIds::AbstractVector{Int}, sanityCheck::Bool=true) ->
    Tuple{Vector{Int}, Vector{Int}}

One greedy pass: starting from the fragment `fragIds` — validated, under `sanityCheck`
(opt-out), to induce a connected line graph (the empty fragment qualifies) — offer
every candidate in the order of `candIds`, which must be disjoint from `fragIds`. A
candidate with no frustration edge into the fragment (while it is non-empty) is
*deferred* into the second returned vector, staying eligible for later passes; a
candidate whose admission keeps the frustration graph a connected line graph is
accepted; any other candidate is discarded permanently, heredity making the rejection
final. Returns the fragment extended in acceptance order and the deferred candidates in
visiting order.
"""
function expandConnectedLineGraph(strs::AbstractVector{PauliStr},
                                  fragIds::AbstractVector{Int},
                                  candIds::AbstractVector{Int}, sanityCheck::Bool=true)
    checkTermIds(strs, vcat(fragIds, candIds)) #> Enforces set disjointness
    edges = NTuple{2, Int}[] #> Edges among accepted terms, labeled by acceptance order
    accepted = collect(fragIds)
    nFrag = length(accepted)

    for j in 2:nFrag, i in 1:(j - 1) #> Form the fragment-internal frustration edges
        checkAntiCom(strs[accepted[begin+i-1]], strs[accepted[begin+j-1]]) &&
        push!(edges, (i, j))
    end

    sanityCheck && isConnectedLineGraph(StrictGraph(nFrag, edges), true)

    deferred = Int[]
    for idx in candIds
        cand = strs[idx]
        nAcc = length(accepted)
        nEdgeOld = length(edges)
        for pos in 1:nAcc
            checkAntiCom(strs[accepted[begin+pos-1]], cand) && push!(edges, (pos, nAcc+1))
        end
        if length(edges) == nEdgeOld && nAcc > 0
            push!(deferred, idx) #> No edge into the fragment: deferred, not discarded
        elseif isConnectedLineGraph(StrictGraph(nAcc + 1, edges))
            push!(accepted, idx)
        else
            resize!(edges, nEdgeOld) #> Roll back the last candidate's edges
        end
    end

    (accepted, deferred)
end

expandConnectedLineGraph

## The Search (`Order` and `Search`)

`findMaxConnectedLGraphSet` splits the terms into `S_diag`, `S_quad`, and the rest,
`S_rest`. A support of size two is an edge between two Majorana indices, and two
such terms anticommute iff their edges share exactly one endpoint; the frustration
graph on `S_quad` is therefore a line graph, and each of its connected components
seeds one branch of the search. One more seed is grown greedily from `S_rest`. A
branch expands its seed in two passes — over `S_rest ∪ S_diag` and then over the
rest of `S_quad`, or, for the last seed, over `S_quad ∪ S_diag` and then over the
deferred part of `S_rest` — and re-offers its deferred terms until a full sweep
accepts none. At that point no single term of the Hamiltonian can be added to the
fragment.

Candidates are visited in one of two orders (`Order`). The *support-descending*
order (`descendCoeff = false`) visits them by descending `κ` and consults no
coefficient, so every branch is a function of the supports alone. The
*coefficient-descending* order (`descendCoeff = true`) visits them by descending
coefficient magnitude, exact ties keeping the support-descending order. The
branches are finally ranked by `(term count, l1 norm)` or, with
`rankByRatio = true`, by `(l1 norm, term count)`; exact ties go to the
lexicographically larger sequence of sorted support keys, then to the earlier
branch. The l1 norms compared in the ranking are compensated double-precision sums
taken in acceptance order, and two branches tie when these sums are equal.

In [5]:
"""
    evalCoeffRatio(ham::PauliSum{T}, termIds::AbstractVector{Int},
                   denominator::Union{T, Missing}=missing) where {T<:Real} -> T

The l1 coefficient ratio of the term set `termIds`: the Neumaier sum of the selected
coefficient magnitudes — accumulated in the order of `termIds` — over `denominator`,
which defaults to the Neumaier sum of all coefficient magnitudes of `ham` in canonical
order (identity-excluded whenever `ham` was loaded with `excludeIdentity=true`).
Passing `denominator = one(T)` turns the function into a plain compensated sum, which
is how the search's final ranking uses it. Both sums are order-sensitive at the level
of the last few ulps.
"""
function evalCoeffRatio(ham::PauliSum{T}, termIds::AbstractVector{Int},
                        denominator::Union{T, Missing}=missing) where {T<:Real}
    checkTermIds(ham.str, termIds)

    numerator = neumaierSum(evalMagnitude, T, view(ham.coeff, termIds))
    ismissing(denominator) && (denominator = neumaierSum(evalMagnitude, T, ham.coeff))

    iszero(denominator) ? zero(T) : numerator / denominator
end

"""
    evalTermSetState(ham::PauliSum, termIds::AbstractVector{Int}) -> NTuple{3, Real}

The state `(count, count ratio, l1 ratio)` of the term set `termIds`, both ratios
taken against all terms of `ham` (see [`evalCoeffRatio`](@ref)).
"""
function evalTermSetState(ham::PauliSum, termIds::AbstractVector{Int})
    nTerm = length(termIds)
    (nTerm, nTerm / countTerms(ham), evalCoeffRatio(ham, termIds))
end

"""
    findMaxConnectedLGraphSet(ham::PauliSum{T}, majFrame::AbstractVector{PauliStr},
                              rankByRatio::Bool=false; descendCoeff::Bool=false,
                              sanityCheck::Bool=true, checkResidual::Bool=false
                              ) where {T<:Real} ->
    Tuple{Vector{Int}, NamedTuple}

Search `ham`, which must carry no identity term, for a maximal connected line-graph
fragment: a term set whose frustration graph is a connected line graph and to which no
single term of `ham` can be added. The terms are split by their Majorana supports (see
[`genMajoranaMasks`](@ref)) into the diagonal block `S_diag` (paired supports), the
quadratic sector `S_quad` (supports of size two), and the rest `S_rest`. Every
connected component of the frustration graph on `S_quad` seeds one branch, and so does
one greedy scan through `S_rest`; each branch runs two passes of
[`expandConnectedLineGraph`](@ref) and then re-offers its deferred terms until a full
sweep accepts none. The best branch is returned.

`descendCoeff` selects the visiting order of every candidate sequence. By default the
terms are visited in descending order of their support keys `(size, mask)` — a total
order, the masks compared as binary numbers with frame position 1 as the lowest bit —
so that every branch is determined by the supports alone. With `descendCoeff = true`
the terms are visited by descending coefficient magnitude, exact ties keeping the
default order. `rankByRatio` acts on the final ranking only: the branches are compared
lexicographically on `(count, coefficient sum)`, the pair swapped when true; exact ties
resolve through the sorted support keys, then in favor of the earlier branch. Every
compared sum accumulates the coefficient magnitudes in acceptance order (see
[`evalCoeffRatio`](@ref)).

`sanityCheck` (opt-out) validates the frame, the supports, every fragment entering a
pass, and the identification of `S_diag` with the diagonal strings. `checkResidual`
(opt-in) re-offers every unselected term to the returned fragment and warns if any is
addable.

The first returned value lists the terms of the winning fragment in acceptance order;
the second records `(branch, seed, pass1, pass2, fixedpoint, supports)`: `branch` is
the index `w` of the winning seed among the `K + 1` seeds (`w <= K` for the `w`-th
component of `S_quad`, `K + 1` for the `S_rest` scan), the next four entries hold the
states of the fragment after each stage (see [`evalTermSetState`](@ref)), and
`supports` is the sorted mask fingerprint of the fragment — the object two encodings of
one Hamiltonian must share.
"""
function findMaxConnectedLGraphSet(ham::PauliSum{T}, majFrame::AbstractVector{PauliStr},
                                   rankByRatio::Bool=false; descendCoeff::Bool=false,
                                   sanityCheck::Bool=true, checkResidual::Bool=false
                                   ) where {T<:Real}
    strs = ham.str
    any(isIdentity, strs) && throw(ArgumentError("`ham` must carry no identity term."))
    masks = genMajoranaMasks(ham, majFrame, sanityCheck)

    sDiag = listDiagBlock(masks)
    sQuad = listQuadSector(masks)
    sRest = setdiff(eachindex(masks), sDiag, sQuad)

    if sanityCheck && sDiag != findall(str -> all(iszero, str.x), strs)
        throw(AssertionError("The paired supports do not reproduce the diagonal block."))
    end

    maskKeys = map(mask -> (sum(mask), reverse(mask)), masks)
    maskKey(i::Int) = maskKeys[begin+i-1]
    sort!(sDiag, by=maskKey)
    sort!(sQuad, by=maskKey)
    sort!(sRest, by=maskKey)

    function reorderTermIds(ids::AbstractVector{Int})
        order = sort(ids, by=maskKey, rev=true) #> The support-descending order
        if descendCoeff #> Stable: exact magnitude ties keep the support-descending order
            sort!(order, by=(i -> evalMagnitude(ham.coeff[i])), rev=true,
                  alg=Base.Sort.DEFAULT_STABLE)
        end
        order
    end

    seeds = listConnectedComponents(strs, sQuad)
    nSeedQuad = length(seeds)
    seedRest, deferRest = expandConnectedLineGraph(strs, Int[], reorderTermIds(sRest),
                                                   sanityCheck)
    push!(seeds, seedRest)

    outcomes = NamedTuple[]
    for (w, seed) in enumerate(seeds)
        pool1, pool2 = if w > nSeedQuad #> The last branch: the greedy `S_rest` scan
            (sQuad, deferRest)
        else
            (sRest, setdiff(sQuad, seed))
        end

        candIds1 = reorderTermIds(setdiff(union(pool1, sDiag), seed))
        pass1Ids, defer1 = expandConnectedLineGraph(strs, seed, candIds1, sanityCheck)
        candIds2 = reorderTermIds(setdiff(pool2, pass1Ids))
        pass2Ids, defer2 = expandConnectedLineGraph(strs, pass1Ids, candIds2, sanityCheck)

        #> Each sweep returns its deferred terms in visiting order, so the re-offering
        #> sequence needs no re-sorting to stay in the configured order
        selected = pass2Ids
        candIds = reorderTermIds(setdiff(union(defer1, defer2), selected))
        while !isempty(candIds)
            grown, candIds = expandConnectedLineGraph(strs, selected, candIds,
                                                      sanityCheck)
            length(grown) == length(selected) && break #> Fixed point reached
            selected = grown
        end

        push!(outcomes, (branch=w, seedIds=seed, pass1Ids=pass1Ids, pass2Ids=pass2Ids,
                         selected=selected))
    end

    function getRank(res)
        countKey = T(length(res.selected))
        coeffKey = evalCoeffRatio(ham, res.selected, one(T))
        keySeq = sort!(map(maskKey, res.selected))
        rankByRatio ? (coeffKey, countKey, keySeq) : (countKey, coeffKey, keySeq)
    end

    winner = argmax(getRank, outcomes) #> The earliest branch among the maximal ones

    if checkResidual
        fragIds = winner.selected
        residual = count(setdiff(eachindex(masks), fragIds)) do i
            extended = first(expandConnectedLineGraph(strs, fragIds, [i], sanityCheck))
            length(extended) > length(fragIds)
        end
        residual > 0 &&
        @warn "The returned fragment is not maximal: $residual term(s) remain addable."
    end

    info = (branch=winner.branch, seed=evalTermSetState(ham, winner.seedIds),
            pass1=evalTermSetState(ham, winner.pass1Ids),
            pass2=evalTermSetState(ham, winner.pass2Ids),
            fixedpoint=evalTermSetState(ham, winner.selected),
            supports=sort!([masks[begin+i-1] for i in winner.selected]))
    (winner.selected, info)
end

findMaxConnectedLGraphSet

## The Structure Table (`tab:sys-db-1body`)

`genStructureTable` evaluates, for each Hamiltonian, the number of qubits `n`, the
number of non-identity terms `|V|`, and three term sets: `S_diag`, `S_quad`, and
the largest connected component of the frustration graph on `S_quad`. Each set is
reported as `count (count ratio, l1 ratio)`, both ratios taken against the
identity-free Hamiltonian. Where two components of `S_quad` tie for the largest
size — the two spin sectors of H4, LiH, and H2O — the first one in `κ`-ascending
order is reported; the tied components share the same entry at this precision.

The cells below define the table utilities and fetch the files.

In [6]:
"""
    formatState(state::NTuple{3, Real}) -> String

Format a state `(count, count ratio, l1 ratio)` as `"count (count ratio, l1 ratio)"`
with three decimal places, the format of the table entries.
"""
function formatState(state::NTuple{3, Real})
    nTerm, countRatio, coeffRatio = state
    @sprintf("%d (%.3f, %.3f)", nTerm, countRatio, coeffRatio)
end

"""
    genSummaryTable(headers::NTuple{N, AbstractString},
                    rows::AbstractVector{<:NTuple{N, Any}}) where {N} -> String

Render `rows` as an aligned text table under `headers`, the leading column left-aligned
and all others right-aligned.
"""
function genSummaryTable(headers::NTuple{N, AbstractString},
                         rows::AbstractVector{<:NTuple{N, Any}}) where {N}
    isempty(rows) && throw(ArgumentError("`rows` must be non-empty."))
    cells = [map(string, row) for row in rows]
    pushfirst!(cells, map(string, headers))
    widths = ntuple(c -> maximum(row -> textwidth(row[c]), cells), N)

    lines = map(cells) do row
        padded = ntuple(c -> (c == 1 ? rpad : lpad)(row[c], widths[c]), N)
        join(padded, "  ")
    end
    insert!(lines, 2, join(map(w -> "-"^w, widths), "  "))

    join(lines, "\n") * "\n"
end

dataDir = joinpath(@__DIR__, "symmerHamData")
pathsJW, pathsBK = map(("JW", "BK")) do encoding
    names = [system * "_STO-3G_SINGLET_" * encoding * ".json"
             for system in CONSTVAR!!systemNames]
    fetchSymmerHamFiles(dataDir, names)
end;

In [7]:
"""
    genStructureTable(paths::AbstractVector{<:AbstractString}) ->
    Vector{Tuple{String, Int, Int, String, String, String}}

One row `(system, n, |V|, S_diag, S_quad, connected S_quad)` per Hamiltonian file in
`paths`: the site count, the number of non-identity terms, and the formatted states
(see [`formatState`](@ref)) of the diagonal block, of the quadratic sector, and of the
first maximum-size connected component of the frustration graph on the quadratic
sector, the components taken in ascending order of their support keys.
"""
function genStructureTable(paths::AbstractVector{<:AbstractString})
    map(paths) do path
        ham = loadSymmerHam(path, excludeIdentity=true)
        name = (first∘splitext∘basename)(path)
        masks = genMajoranaMasks(ham, genSymmerMajFrame(name, countSites(ham)))
        maskKey(i::Int) = let mask=masks[begin+i-1]; (sum(mask), reverse(mask)) end

        sDiag = listDiagBlock(masks)
        sQuad = sort!(listQuadSector(masks), by=maskKey)
        components = listConnectedComponents(ham.str, sQuad)
        connQuad = isempty(components) ? Int[] : argmax(length, components)

        formatSet(ids) = formatState(evalTermSetState(ham, ids))
        (String(first(split(name, '_'))), countSites(ham), countTerms(ham),
         formatSet(sDiag), formatSet(sQuad), formatSet(connQuad))
    end
end

structureHeaders = ("System", "n", "|V|", "S_diag", "S_quad", "Connected S_quad")
structureRows = genStructureTable(pathsJW)

print(genSummaryTable(structureHeaders, structureRows))

System   n   |V|              S_diag             S_quad   Connected S_quad
------  --  ----  ------------------  -----------------  -----------------
H3+      6    51   21 (0.412, 0.791)   8 (0.157, 0.247)   4 (0.078, 0.092)
H4       8   104   36 (0.346, 0.570)  12 (0.115, 0.092)   4 (0.038, 0.027)
LiH     12   630   78 (0.124, 0.794)  36 (0.057, 0.375)  16 (0.025, 0.150)
H2O     14  1085  105 (0.097, 0.773)  42 (0.039, 0.612)  16 (0.015, 0.249)


## The Search Table (`tab:line-graph-search`)

`genSearchTable` runs the search on each Hamiltonian under its four
configurations, the two rankings times the two visiting orders; a column header
reads `ranking/visiting order`. Every returned fragment is re-certified as a
connected line graph, and every unselected term is re-offered to it
(`checkResidual`) to confirm that none can be added.

In [8]:
"""
    genSearchTable(paths::AbstractVector{<:AbstractString}; checkResidual::Bool=true) ->
    Tuple{Vector{Tuple{String, Int, String, String, String, String}}, Dict}

Run [`findMaxConnectedLGraphSet`](@ref) on every Hamiltonian file in `paths` under its
four configurations and return one row
`(system, |V|, count/support, count/coeff, l1/support, l1/coeff)` per file, each of the
last four entries being the formatted state (see [`formatState`](@ref)) of the fragment
found under the named `ranking/visiting order`. The second returned value maps
`(system, rankByRatio, descendCoeff)` to the support fingerprint of that fragment.
Every fragment is re-certified as a connected line graph; `checkResidual` (opt-out)
additionally re-offers every unselected term to it.
"""
function genSearchTable(paths::AbstractVector{<:AbstractString}; checkResidual::Bool=true)
    rows = Tuple{String, Int, String, String, String, String}[]
    supports = Dict{Tuple{String, Bool, Bool}, Vector{BitVector}}()

    for path in paths
        ham = loadSymmerHam(path, excludeIdentity=true)
        name = (first∘splitext∘basename)(path)
        system = String(first(split(name, '_')))
        majFrame = genSymmerMajFrame(name, countSites(ham))

        states = String[]
        for rankByRatio in (false, true), descendCoeff in (false, true)
            ids, info = findMaxConnectedLGraphSet(ham, majFrame, rankByRatio;
                                                  descendCoeff, checkResidual)
            edges = getFrustrationInfo(ham.str[ids]).second
            isConnectedLineGraph(StrictGraph(length(ids), edges), true)

            push!(states, formatState(info.fixedpoint))
            supports[(system, rankByRatio, descendCoeff)] = info.supports
        end

        push!(rows, (system, countTerms(ham), states...))
    end

    (rows, supports)
end

searchHeaders = ("System", "|V|", "count/support", "count/coeff", "l1/support", "l1/coeff")
searchRows, searchSupports = genSearchTable(pathsJW)

print(genSummaryTable(searchHeaders, searchRows))

System   |V|      count/support        count/coeff         l1/support           l1/coeff
------  ----  -----------------  -----------------  -----------------  -----------------
H3+       51  16 (0.314, 0.223)  16 (0.314, 0.269)  16 (0.314, 0.223)  16 (0.314, 0.269)
H4       104  22 (0.212, 0.194)  22 (0.212, 0.194)  22 (0.212, 0.194)  22 (0.212, 0.194)
LiH      630  96 (0.152, 0.320)  50 (0.079, 0.163)  96 (0.152, 0.320)  50 (0.079, 0.163)
H2O     1085  91 (0.084, 0.523)  37 (0.034, 0.254)  91 (0.084, 0.523)  37 (0.034, 0.254)


## Encoding Invariance

The Bravyi-Kitaev files encode the same four Hamiltonians. The cell below asserts
that they reproduce both tables and that, under the support-descending order, the
fragments themselves are identical, support for support. The
coefficient-descending order carries no such guarantee: the two files of a
molecule store coefficients that differ in their last bits (by less than `1e-12`
relative here), and such a difference can reorder terms of equal or nearly equal
magnitude in a visiting sequence. For H4 it exchanges two occupation terms whose
magnitudes are bit-equal in one file and 8 ulps apart in the other — 21 of 22
supports shared — without changing the table entry.

In [9]:
structureRowsBK = genStructureTable(pathsBK)
searchRowsBK, searchSupportsBK = genSearchTable(pathsBK)

structureRowsBK == structureRows ||
error("The structure table changes under the Bravyi-Kitaev encoding.")
searchRowsBK == searchRows ||
error("The search table changes under the Bravyi-Kitaev encoding.")

configs = [(system, rankByRatio, descendCoeff) for system in CONSTVAR!!systemNames
           for rankByRatio in (false, true) for descendCoeff in (false, true)]
differing = filter(key -> searchSupports[key] != searchSupportsBK[key], configs)

for (system, rankByRatio, descendCoeff) in differing
    #> The support-descending order consults no coefficient before the final ranking
    descendCoeff || error("The fragment of ", system, " under the support-descending ",
                          "order changes with the encoding.")
    fpJW = searchSupports[(system, rankByRatio, descendCoeff)]
    fpBK = searchSupportsBK[(system, rankByRatio, descendCoeff)]
    println(system, ", ", rankByRatio ? "l1" : "count", " ranking, coefficient-",
            "descending order: ", length(intersect(fpJW, fpBK)), " of ", length(fpJW),
            " (JW) / ", length(fpBK), " (BK) supports shared")
end
println("Both tables are identical under the two encodings; the fragments themselves ",
        "coincide in ", length(configs) - length(differing), " of ", length(configs),
        " configurations.")

H4, count ranking, coefficient-descending order: 21 of 22 (JW) / 22 (BK) supports shared
H4, l1 ranking, coefficient-descending order: 21 of 22 (JW) / 22 (BK) supports shared
Both tables are identical under the two encodings; the fragments themselves coincide in 14 of 16 configurations.
